# Notebook per esportare pickle per streamlit 

Per i dataset Titanic e German vengono esportati:
* X train per le instance
* Explainer
* bbox - Il modello è un random forest
* La feature importance con Shap
* 20 istanze per dataset già spiegate


In [1]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt

import os
from plot_explanation import PlotExplanation
import pickle

In [2]:
datasets=['titanic_c.csv','german_credit.csv']

## Titanic

In [61]:
source_file = f'../datasets/{datasets[0]}'
class_field = 'Survived'
# Load and transform dataset
df_t = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [62]:
df_t, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df_t, class_field)

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [63]:
test_size = 0.3
random_state = 42
X_train_t, X_test_t, Y_train_t, Y_test_t = train_test_split(df_t[feature_names], df_t[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df_t[class_field])

In [64]:
bb_t = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb_t.fit(X_train_t.values, Y_train_t.values)
bbox_t = sklearn_classifier_wrapper(bb_t)

### Export Pickle for X_train and bbox

In [65]:
dataset_dict_tit = {"X_train_titanic": X_train_t, "X_test_titanic": X_test_t, "y_train_titanic": Y_train_t, "y_test_titanic": Y_test_t}
path="../datasets/titanic/train_test_titanic.pkl"

In [66]:
with open(path, 'wb') as train_test_titanic:
    pickle.dump(dataset_dict_tit, train_test_titanic)

In [67]:
pickle.dump(bbox_t, open('../datasets/titanic/bbox_titanic.pkl', 'wb'))

In [68]:
inst= X_train_t.iloc[4].values
inst

array([ 3.  ,  1.  , 28.  ,  1.  ,  0.  , 15.85,  2.  ])

In [69]:
print('Instance ',inst)
print('True class ',Y_train_t.iloc[4])
print('Predicted class ',bb_t.predict(inst.reshape(1, -1)))

Instance  [ 3.    1.   28.    1.    0.   15.85  2.  ]
True class  0
Predicted class  [0]


In [12]:
inst

array([ 3.  ,  1.  , 28.  ,  1.  ,  0.  , 15.85,  2.  ])

### Export the explainer

In [70]:
explainer_t = LoreTabularExplainer(bbox_t)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer_t.fit(df_t, class_field, config)
#pickle.dump(explainer_t, open('../datasets/titanic/explainer_titanic.pkl', 'wb'))

In [26]:
explainer_t.explain(inst)

### Export the feature Importance

In [71]:
explainer_s_t = ShapXAITabularExplainer(bbox_t, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train_t.iloc[0:100].values}
explainer_s_t.fit(config)
#pickle.dump(explainer_s_t, open('../datasets/titanic/shap_featureimportance_titanic.pkl', 'wb'))

In [72]:
exp_t = explainer_s_t.explain(inst)

In [73]:
shap_feature_importance_t=exp_t.exp

## Export the pickle with instances explained

In [17]:
#list_of_inst_t=X_train_t.iloc[0:2].values

In [74]:
dfnew_t= X_train_t.iloc[0:5]

In [ ]:
row_list=[]
list_exp_inst_t=[]
exp_list_t=[]
elem_list_3_t=[]
shap_list_t=[]
n=0
for index, row in dfnew_t.iterrows():
    row_dict = {}
    #dizionario per riga del df
    #row_list.append(row.tolist()) #creo una lista delle istanze da spiegare
    #for inst_t in row_list: #per ogni instanza della lista
    exp = explainer_t.explain(row.values) #spiego l'istanza, explainer t è stato trainato prima
    exp_list_t.append(exp) # la appendo a una nuova lista
    for col in dfnew_t.columns: # per ogni colonna del df
        row_dict[col] = row[col]# creo un dizionario
    row_list.append(row_dict)
    # feature importance con shap
    exp_shap_t = explainer_s_t.explain(row)
    shap_feature_importance_t = np.array(shap_feature_importance_t)
    shap_list_t.append(shap_feature_importance_t)
    # appendo i 3 elementi in una lista
    elem_list_3_t = [exp_list_t[-1], row_dict, shap_list_t[-1]]
    list_exp_inst_t.append(elem_list_3_t) #lista finale
    n+=1
    print(exp)
    print(n)

1
2
3
4


[1.0, 1.0, 19.0, 1.0, 0.0, 53.1, 2.0]

In [29]:
print(list_exp_inst_t)

[[<xailib.explainers.lore_explainer.LoreTabularExplanation object at 0x7ff0f7378e50>, {'Pclass': 1.0, 'Sex': 1.0, 'Age': 19.0, 'SibSp': 1.0, 'Parch': 0.0, 'Fare': 53.1, 'Embarked': 2.0}, array([[ 0.05785741,  0.10758353,  0.0709937 ,  0.01493528,  0.01607778,
         0.07587915,  0.03636818],
       [-0.05785741, -0.10758353, -0.0709937 , -0.01493528, -0.01607778,
        -0.07587915, -0.03636818]])]]


In [ ]:
pickle.dump(list_exp_inst_t, open('../datasets/titanic/instances_explained_titanic.pkl', 'wb'))

In [ ]:
inst_dict=X_train_t.iloc[0:2].to_dict('index')

In [ ]:
exp.expDict['rule']['premise']

In [ ]:
rules =exp.expDict['rule']['premise']

In [ ]:
for r in rules:
    print(r.values())

In [ ]:
print(exp.expDict['rule']['premise'])

In [ ]:
#pickle.dump(list_exp_t, open('../datasets/titanic/instances_explained_titanic.pkl', 'wb'))

## German

In [31]:
source_file = f'../datasets/{datasets[1]}'
class_field = 'default'
# Load and transform dataset
df_g = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [32]:
df_g, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df_g, class_field)

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [33]:
test_size = 0.3
random_state = 42
X_train_g, X_test_g, Y_train_g, Y_test_g= train_test_split(df_g[feature_names], df_g[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df_g[class_field])

In [34]:
bb_g = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb_g.fit(X_train_g.values, Y_train_g.values)
bbox_g = sklearn_classifier_wrapper(bb_g)

### Export Pickle for X_train and bbox

In [35]:
dataset_dict_ger = {"X_train_german": X_train_g, "X_test_german": X_test_g, "y_train_german": Y_train_g, "y_test_german": Y_test_g}
path="../datasets/german/train_test_german.pkl"
with open(path, 'wb') as train_test_german:
    pickle.dump(dataset_dict_ger, train_test_german)

In [36]:
pickle.dump(bbox_g, open('../datasets/german/bbox_german.pkl', 'wb'))

In [37]:
inst_g = X_train_g.iloc[4].values
print('Instance ',inst_g)
print('True class ',Y_train_g.iloc[4])
print('Predicted class ',bb_g.predict(inst_g.reshape(1, -1)))

Instance  [  11 7228    1    4   39    2    1    0    0    0    1    0    1    0
    0    0    0    0    1    0    0    0    0    0    0    0    0    1
    0    0    0    0    0    1    0    0    0    0    0    1    0    0
    1    1    0    0    0    0    1    0    0    1    0    0    0    0
    1    1    0    0    1]
True class  0
Predicted class  [0]


### Export the explainer

In [38]:
explainer_g = LoreTabularExplainer(bbox_g)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer_g.fit(df_g, class_field, config)
pickle.dump(explainer_g, open('../datasets/german/explainer_german.pkl', 'wb'))


In [39]:
exp = explainer_g.explain(inst_g)
print(exp)

### Export the feature Importance

In [40]:
explainer_s_g = ShapXAITabularExplainer(bbox_g, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train_g.iloc[0:100].values}
explainer_s_g.fit(config)
#pickle.dump(explainer_s_g, open('../datasets/german/shap_featureimportance_german.pkl', 'wb'))

In [41]:
exp_g = explainer_s_g.explain(inst_g)

In [42]:
shap_feature_importance=exp_g.exp

## Export the pickle with instances explained

In [43]:
list_of_inst_g=X_train_g.iloc[0:20].values

In [44]:
list_of_inst_g[2]

array([  18, 4165,    2,    2,   36,    2,    2,    0,    0,    0,    1,
          0,    0,    0,    0,    1,    0,    1,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    1,    0,    0,    0,    0,    0,
          1,    0,    0,    0,    0,    0,    1,    0,    0,    1,    0,
          1,    0,    0,    0,    0,    1,    0,    1,    0,    0,    1,
          0,    0,    1,    0,    0,    1])

In [45]:
#list_exp_g=[]
#for inst_g in list_of_inst_g:
#    exp = explainer_g.explain(inst_g)
#    list_exp_g.append(exp)
#    print(exp)

KeyboardInterrupt: 

In [ ]:
list_exp_g[1].getExemplars

In [ ]:
list_exp_g[1].expDict['rule']['premise']

In [ ]:
pickle.dump(list_exp_g, open('../datasets/german/instances_explained_german.pkl', 'wb'))

## Export the instances explained

In [46]:
dfnew_g= X_train_g.iloc[0:5]

In [47]:
row_list_g=[]
list_exp_inst_g=[]
exp_list_g=[]
elem_list_3_g=[]
shap_list_g=[]
n=0
for index, row in dfnew_g.iterrows():
    row_dict = {} #dizionario per riga del df
    #row_list_g.append(row.tolist()) #creo una lista delle istanze da spiegare
   #per ogni instanza della lista
    exp = explainer_g.explain(row.values) #spiego l'istanza, explainer g è stato trainato prima
    exp_list_g.append(exp) # la appendo a una nuova lista
    for col in dfnew_g.columns: # per ogni colonna del df
        row_dict[col] = row[col]# creo un dizionario
    row_list_g.append(row_dict)
    # feature importance con shap
    exp_shap_g = explainer_s_g.explain(row)
    shap_feature_importance_g=exp_shap_g.exp
    shap_list_g.append(shap_feature_importance_g)
    # appendo i 3 elementi in una lista
    elem_list_3_g = [exp_list_g[-1], row_dict, shap_list_g[-1]]
    list_exp_inst_g.append(elem_list_3_g) #lista finale
    n+=1
    print(exp)
    print(n)

1
2
3
4
5


In [57]:
print(list_exp_inst_g)

[[<xailib.explainers.lore_explainer.LoreTabularExplanation object at 0x7ff0f74590d0>, {'duration_in_month': 12, 'credit_amount': 1295, 'installment_as_income_perc': 3, 'present_res_since': 1, 'age': 25, 'credits_this_bank': 1, 'people_under_maintenance': 1, 'account_check_status=0 <= ... < 200 DM': 1, 'account_check_status=< 0 DM': 0, 'account_check_status=>= 200 DM / salary assignments for at least 1 year': 0, 'account_check_status=no checking account': 0, 'credit_history=all credits at this bank paid back duly': 0, 'credit_history=critical account/ other credits existing (not at this bank)': 0, 'credit_history=delay in paying off in the past': 0, 'credit_history=existing credits paid back duly till now': 1, 'credit_history=no credits taken/ all credits paid back duly': 0, 'purpose=(vacation - does not exist?)': 0, 'purpose=business': 0, 'purpose=car (new)': 1, 'purpose=car (used)': 0, 'purpose=domestic appliances': 0, 'purpose=education': 0, 'purpose=furniture/equipment': 0, 'purpose

In [59]:
pickle.dump(list_exp_inst_g, open('../datasets/german/instances_explained_german.pkl', 'wb'))